In [1]:
# ── CELL 1: Install ───────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q", "groq"], check=True)
print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 6.0 MB/s eta 0:00:00
✓ Dependencies installed


In [2]:
# ── CELL 2: Imports ───────────────────────────────────────────
import os
import json
import pickle
import time
from groq import Groq
from collections import defaultdict
from kaggle_secrets import UserSecretsClient
print("✓ Imports done")

✓ Imports done


In [4]:
# ── CELL 3: Load secrets ──────────────────────────────────────
secrets  = UserSecretsClient()
GROQ_KEY = secrets.get_secret("GROQ_API_KEY")
client   = Groq(api_key=GROQ_KEY)
print("✓ Groq client ready")
print("  Judge model: llama-3.3-70b-versatile")

✓ Groq client ready
  Judge model: llama-3.3-70b-versatile


In [6]:
# ── CELL 4: Load production corpus ────────────────────────────
CORPUS_PATH = "/kaggle/input/datasets/rockybhai159/hello123/corpus_B_recursive.pkl"

with open(CORPUS_PATH, "rb") as f:
    data = pickle.load(f)

chunks   = data["chunks"]
metadata = data["metadata"]
print(f"✓ Loaded {len(chunks)} chunks")

✓ Loaded 2753 chunks


In [7]:
# ── CELL 5: Group chunks by paper ─────────────────────────────
paper_chunks = defaultdict(list)
for chunk, meta in zip(chunks, metadata):
    paper_chunks[meta["title"]].append(chunk)

print(f"✓ Found {len(paper_chunks)} papers:")
for title in paper_chunks:
    print(f"   {title[:65]}  →  {len(paper_chunks[title])} chunks")

✓ Found 15 papers:
   Nested Learning: The Illusion of Deep Learning Architectures  →  495 chunks
   Fast Weight Programming and Linear Transformers: From ML to Neuro  →  258 chunks
   End-to-End Test-Time Training for Long Context  →  214 chunks
   Dynamic Nested Hierarchies: Self-Evolution in ML for Lifelong Int  →  103 chunks
   CRAD-HOPE: Brain-inspired Nested Learning Framework for Few-Shot   →  177 chunks
   Tiny Autoregressive Recursive Models  →  96 chunks
   HYDRA: Dual Exponentiated Memory for Multivariate Time Series Ana  →  121 chunks
   Beyond Test-Time Training: Learning to Reason via Hardware-Effici  →  160 chunks
   AllMem: A Memory-centric Recipe for Efficient Long-context Modeli  →  103 chunks
   Vision Transformers That Never Stop Learning  →  102 chunks
   MSA: Memory Sparse Attention for Efficient End-to-End Memory Mode  →  159 chunks
   Prompt Injection Mitigation with Agentic AI, Nested Learning, and  →  226 chunks
   Memory Caching: RNNs with Growing Memory  →  

In [8]:
# ── CELL 6: Generate Q&A pairs ────────────────────────────────
print("\nGenerating golden Q&A pairs using llama-3.3-70b-versatile...\n")

golden = []
failed = []

for title, paper_chunk_list in paper_chunks.items():
    # Take first 4 chunks — abstract + intro = richest content
    sample = " ".join(paper_chunk_list[:4])[:2500]

    prompt = f"""You are an expert AI researcher specializing in machine learning.

Based on the following excerpt from an academic paper, generate exactly 2 high-quality technical questions with detailed answers.

STRICT RULES:
- Questions must be answerable ONLY from the provided text
- Questions must require reasoning and understanding, not just copying a sentence
- Answers must be detailed, specific, and grounded in the text
- Do NOT ask trivial questions like "what is the paper about"
- Focus on concepts like: how methods work, why design choices were made, what limitations exist, how this relates to Nested Learning
- Return ONLY valid JSON, no markdown, no backticks, no extra text
- Format exactly: [{{"question": "...", "answer": "..."}}]

Paper Title: {title}

Excerpt:
{sample}"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=800,
        )
        text = response.choices[0].message.content.strip()

        # Clean any accidental markdown
        text = text.replace("```json", "").replace("```", "").strip()

        qas = json.loads(text)

        # Validate and tag each Q&A
        valid = []
        for qa in qas:
            if "question" in qa and "answer" in qa:
                if len(qa["question"]) > 20 and len(qa["answer"]) > 50:
                    qa["paper"] = title
                    valid.append(qa)

        golden.extend(valid)
        print(f"  ✓ {title[:60]}  →  {len(valid)} questions")
        time.sleep(1.5)   # avoid Groq rate limits

    except json.JSONDecodeError as e:
        print(f"  ✗ JSON error for {title[:50]}: {e}")
        failed.append(title)
    except Exception as e:
        print(f"  ✗ Error for {title[:50]}: {e}")
        failed.append(title)
        time.sleep(3)     # wait longer after errors


Generating golden Q&A pairs using llama-3.3-70b-versatile...

  ✓ Nested Learning: The Illusion of Deep Learning Architectures  →  2 questions
  ✓ Fast Weight Programming and Linear Transformers: From ML to   →  2 questions
  ✓ End-to-End Test-Time Training for Long Context  →  2 questions
  ✓ Dynamic Nested Hierarchies: Self-Evolution in ML for Lifelon  →  2 questions
  ✓ CRAD-HOPE: Brain-inspired Nested Learning Framework for Few-  →  2 questions
  ✓ Tiny Autoregressive Recursive Models  →  2 questions
  ✓ HYDRA: Dual Exponentiated Memory for Multivariate Time Serie  →  2 questions
  ✓ Beyond Test-Time Training: Learning to Reason via Hardware-E  →  2 questions
  ✓ AllMem: A Memory-centric Recipe for Efficient Long-context M  →  2 questions
  ✓ Vision Transformers That Never Stop Learning  →  2 questions
  ✓ MSA: Memory Sparse Attention for Efficient End-to-End Memory  →  2 questions
  ✓ Prompt Injection Mitigation with Agentic AI, Nested Learning  →  2 questions
  ✓ Memory Caching:

In [9]:
# ── CELL 7: Save golden set ───────────────────────────────────
# Keep exactly 30
golden = golden[:30]

with open("/kaggle/working/golden_set.json", "w") as f:
    json.dump(golden, f, indent=2)

print(f"\n{'='*60}")
print(f"  GOLDEN SET SUMMARY")
print(f"{'='*60}")
print(f"  Total Q&A pairs : {len(golden)}")
print(f"  Failed papers   : {len(failed)}")
if failed:
    for f_title in failed:
        print(f"    - {f_title[:60]}")
print(f"  Saved to        : /kaggle/working/golden_set.json")


# ── CELL 8: Preview all 30 questions ──────────────────────────
print(f"\n{'='*60}")
print(f"  PREVIEW — ALL {len(golden)} QUESTIONS")
print(f"{'='*60}")
for i, qa in enumerate(golden):
    print(f"\n  Q{i+1:02d}: {qa['question']}")
    print(f"  A{i+1:02d}: {qa['answer'][:150]}...")
    print(f"  Paper: {qa['paper'][:55]}")
    print(f"  {'-'*55}")


  GOLDEN SET SUMMARY
  Total Q&A pairs : 30
  Failed papers   : 0
  Saved to        : /kaggle/working/golden_set.json

  PREVIEW — ALL 30 QUESTIONS

  Q01: How does the concept of Nested Learning relate to the idea of in-context learning in large models, and what implications does this have for the design of more expressive learning algorithms?
  A01: According to the excerpt, Nested Learning (NL) suggests that existing deep learning methods learn from data through compressing their own context flow...
  Paper: Nested Learning: The Illusion of Deep Learning Architec
  -------------------------------------------------------

  Q02: What is the significance of representing a machine learning model as a set of nested, multi-level, and/or parallel optimization problems in the context of Nested Learning, and how does this relate to the concept of 'context flow'?
  A02: The excerpt states that Nested Learning coherently represents a machine learning model with a set of nested, multi-level, 